In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [4]:
df1= pd.read_csv('homework_2.1.csv')
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   time    100 non-null    int64  
 1   G1      100 non-null    float64
 2   G2      100 non-null    float64
 3   G3      100 non-null    float64
dtypes: float64(3), int64(1)
memory usage: 3.3 KB


In [12]:

# Convert wide format to long format
df_long = df1.melt(
    id_vars='time',
    value_vars=['G1', 'G2', 'G3'],
    var_name='group',
    value_name='outcome'
)

# Fixed effects regression
model = smf.ols(
    'outcome ~ time + C(group)',
    data=df_long
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                outcome   R-squared:                       0.311
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     44.55
Date:                Thu, 28 May 2026   Prob (F-statistic):           8.72e-24
Time:                        16:20:34   Log-Likelihood:                -216.89
No. Observations:                 300   AIC:                             441.8
Df Residuals:                     296   BIC:                             456.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.0786      0.071      1.

In [13]:
df2= pd.read_csv('homework_2.2.csv')
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       10000 non-null  int64  
 1   Y       10000 non-null  float64
 2   Z       10000 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 234.5 KB


In [14]:

n_bootstrap = 1000
boot_coefs = []

for i in range(n_bootstrap):
    boot_sample = df2.sample(n=len(df2), replace=True, random_state=i)
    model = smf.ols("Y ~ X + Z", data=boot_sample).fit()
    boot_coefs.append(model.params["X"])

boot_coefs = pd.Series(boot_coefs)

print("Bootstrap mean:", boot_coefs.mean())
print("Bootstrap std:", boot_coefs.std())
print("95% CI:", np.percentile(boot_coefs, [2.5, 97.5]))

Bootstrap mean: 2.8126955445200448
Bootstrap std: 0.16453832821814043
95% CI: [2.49498239 3.13349972]


In [15]:
treated_mean = df2[df2["X"] == 1]["Y"].mean()
untreated_mean = df2[df2["X"] == 0]["Y"].mean()

effect = treated_mean - untreated_mean

print(effect)

2.920717264723189


In [17]:
bootstrap_effects = []

n_bootstrap = 1000

for i in range(n_bootstrap):
    
    sample = df2.sample(
        n=len(df2),
        replace=True
    )
    
    treated_mean = sample[sample["X"] == 1]["Y"].mean()
    untreated_mean = sample[sample["X"] == 0]["Y"].mean()
    
    effect = treated_mean - untreated_mean
    
    bootstrap_effects.append(effect)

variance = np.var(bootstrap_effects)

print(variance)

0.0321432330569293


In [18]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import skew

bootstrap_effects = []

n_bootstrap = 1000

for i in range(n_bootstrap):
    
    sample = df2.sample(
        n=len(df2),
        replace=True
    )
    
    model = smf.ols(
        "Y ~ X + Z",
        data=sample
    ).fit()
    
    effect = model.params["X"]
    
    bootstrap_effects.append(effect)

effect_skewness = skew(bootstrap_effects)

print(effect_skewness)

0.09416314605515415
